In [26]:
import sys
import torch
import torch.nn as nn
import torchvision
from tqdm import tqdm
from torchvision import transforms
from torch.utils.data import random_split, DataLoader

device = torch.device("cpu")
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.mps.is_available():
    device = torch.device("mps")

# 데이터 전처리: 이미지를 그대로 사용할 수 없고 텐서로 변경해야 함
preprocess = transforms.Compose([
    transforms.Resize(512),
    transforms.RandomRotation(10),
    transforms.RandomVerticalFlip(),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.2),

    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


dataset = torchvision.datasets.ImageFolder(
    root='./images',
    transform = preprocess # 이미지가 텐서로 변환 됨
)

# print( dataset.class_to_idx )
# print( len(dataset) )
# print( dataset[0] )
# print( dataset[1400] )

train_dataset, val_dataset = random_split( dataset, [0.8, 0.2] ) # 20%는 검증데이터에 사용함

train_dataloader = DataLoader( train_dataset, batch_size=32, shuffle=True )
val_dataloader = DataLoader( val_dataset, batch_size=32, shuffle=True )

resnet50_model = torchvision.models.resnet50(
    weights=torchvision.models.ResNet50_Weights.IMAGENET1K_V1
)

# 이미지를 훈련시킨 resnet50 모델에서 마지막 출력층(레이어)을 잘라내고 나의 새 데이터를 넣어서 훈련하거나
# 가중치와 편향을 조절해야 함 > 전이학습

# resnet50_model.children() # resnet50의 모든 모델을 가져오는 방법
resnet50_model.fc = nn.Identity()  # .fc > 자동으로 마지막 레이어에 덮어쓰는 방법
for param in resnet50_model.parameters():
    param.requires_grad = False  # 마지막 레이어에 대한 기울기 추적(역전파) 하지 않음 > 속도 빨라짐
resnet50_model.eval()
resnet50_model = resnet50_model.to(device)

fc_model = nn.Sequential(
    nn.Linear( 2048, 1024 ),
    nn.ReLU(),
    nn.Linear(1024, 1)
)

fc_model = fc_model.to(device)

model = nn.Sequential(
    resnet50_model,
    fc_model
)
model = model.to(device)

optimizer = torch.optim.Adam( fc_model.parameters(), lr= 0.00025 )
loss_fn = nn.BCEWithLogitsLoss()

for epoch in range(3):
    print(f"--- Epoch:{epoch} ---")
    model.train()
    resnet50_model.eval()

    loss_sum = 0
    train_accurate = 0
    train_sum = 0

    for X, y in tqdm(train_dataloader):
        X = X.to(device)
        y = y.to(device).type( torch.float).reshape(-1,1)

        outputs = model(X)
        optimizer.zero_grad()
        loss = loss_fn( outputs, y )
        loss_sum += loss.item()
        loss.backward()
        optimizer.step()

        predictions = torch.sigmoid(outputs) > 0.5
        accurate = (predictions == y).sum().item()
        train_accurate += accurate
        train_sum += y.size(0)

    print(" Training loss: ", loss_sum / len(train_dataloader) )
    print(" Training accuracy: ", train_accurate / train_sum )

    torch.save( fc_model.state_dict(), f"fc_model_{epoch}.pth" )

    # 모델 평가
    model.eval()
    val_loss_sum = 0
    val_accurate = 0
    val_sum = 0
    with torch.no_grad():
        
        for X,y in tqdm( val_dataloader ):
            X = X.to(device)
            y = y.to(device).type(torch.float).reshape(-1,1)

            outputs = model(X)
            loss = loss_fn(outputs, y)
            val_loss_sum+=loss.item()

            predictions = torch.sigmoid(outputs) > 0.5
            accurate = (predictions == y).sum().item()
            val_accurate += accurate
            val_sum += y.size(0)
    
    print("Validation loss: ", val_loss_sum / len(val_dataloader) )
    print("Validation accuracy: ", val_accurate / val_sum )







--- Epoch:0 ---


100%|██████████| 47/47 [05:49<00:00,  7.43s/it]


 Training loss:  0.3652855119806655
 Training accuracy:  0.8383838383838383


100%|██████████| 12/12 [01:23<00:00,  6.99s/it]


Validation loss:  0.2470385506749153
Validation accuracy:  0.9272237196765498
--- Epoch:1 ---


...

Validation loss:  0.2327696072558562
Validation accuracy:  0.9137466307277629
--- Epoch:2 ---


100%|██████████| 47/47 [05:41<00:00,  7.26s/it]


 Training loss:  0.2001751997369401
 Training accuracy:  0.9151515151515152


100%|██████████| 12/12 [01:25<00:00,  7.10s/it]

Validation loss:  0.1759895651290814
Validation accuracy:  0.9164420485175202
